# BacktestingEngine Visualization with Prometheus & Grafana

This notebook demonstrates how to use Prometheus and Grafana for visualizing backtesting engine metrics and performance. While Grafana/Prometheus are primarily designed for operational monitoring, they can be adapted for backtesting analytics.

**Key Considerations:**
- ✅ **Good for**: Operational monitoring, real-time backtest execution metrics, system performance
- ❌ **Limited for**: Financial analysis, equity curves, detailed trade-by-trade analysis
- 💡 **Best Practice**: Use both - Prometheus/Grafana for operations + Jupyter/Plotly for financial analysis

## Table of Contents
1. Setup Prometheus Client
2. Define Custom Metrics for Backtesting  
3. Instrument Backtesting Code with Metrics
4. Configure Prometheus Scraping
5. Create Grafana Dashboard Programmatically
6. Real-time Backtesting Monitoring

## 1. Setup Prometheus Client

First, let's install and configure the Prometheus Python client library to expose metrics from the backtesting engine.

In [ ]:
# Install required packages
# !pip install prometheus_client requests plotly pandas

# Import Required Libraries
import time
import random
from datetime import datetime, timedelta
from prometheus_client import Counter, Histogram, Gauge, start_http_server, generate_latest
import requests
import json
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

print("✅ Prometheus client and visualization libraries imported successfully")

## 2. Define Custom Metrics for Backtesting

Create custom Prometheus metrics for tracking backtesting performance, including counters, gauges, and histograms for trades, returns, and execution times.

In [ ]:
# Define Prometheus metrics for backtesting

# Strategy Performance Metrics
strategy_total_return = Gauge('backtest_strategy_total_return_percent', 
                             'Total return percentage of the strategy',
                             ['strategy_name', 'symbol', 'timeframe'])

strategy_sharpe_ratio = Gauge('backtest_strategy_sharpe_ratio', 
                             'Sharpe ratio of the strategy',
                             ['strategy_name', 'symbol', 'timeframe'])

strategy_max_drawdown = Gauge('backtest_strategy_max_drawdown_percent', 
                             'Maximum drawdown percentage',
                             ['strategy_name', 'symbol', 'timeframe'])

# Trade Execution Metrics
trades_total = Counter('backtest_trades_total', 
                      'Total number of trades executed',
                      ['strategy_name', 'symbol', 'trade_type'])

trade_pnl = Histogram('backtest_trade_pnl_dollars', 
                     'Individual trade P&L in dollars',
                     ['strategy_name', 'symbol'],
                     buckets=[-1000, -500, -100, -50, -10, 0, 10, 50, 100, 500, 1000, float('inf')])

# Execution Performance Metrics
backtest_execution_time = Histogram('backtest_execution_duration_seconds',
                                   'Time taken to complete backtest',
                                   ['strategy_name', 'symbol', 'timeframe'])

orders_processed_per_second = Gauge('backtest_orders_processed_per_second',
                                   'Number of orders processed per second',
                                   ['strategy_name'])

# Risk Metrics
portfolio_value = Gauge('backtest_portfolio_value_dollars',
                       'Current portfolio value during backtest',
                       ['strategy_name', 'symbol'])

position_size = Gauge('backtest_position_size_shares',
                     'Current position size in shares',
                     ['strategy_name', 'symbol', 'side'])

print("✅ Prometheus metrics defined:")
print("- strategy_total_return: Strategy performance tracking")
print("- trades_total: Trade execution counter")  
print("- trade_pnl: Individual trade P&L distribution")
print("- backtest_execution_time: Performance monitoring")
print("- portfolio_value: Real-time portfolio tracking")

## 3. Instrument Backtesting Code with Metrics

Integrate Prometheus metrics collection into the backtesting engine code to track key performance indicators and strategy metrics.

In [ ]:
class BacktestMetricsCollector:
    """Collects and reports metrics during backtesting execution"""
    
    def __init__(self, strategy_name: str, symbol: str, timeframe: str):
        self.strategy_name = strategy_name
        self.symbol = symbol
        self.timeframe = timeframe
        self.backtest_start_time = time.time()
        self.trade_count = 0
        self.total_pnl = 0.0
        self.peak_value = 0.0
        self.current_drawdown = 0.0
        self.max_drawdown = 0.0
        
    def record_trade(self, trade_type: str, pnl: float, shares: int):
        """Record a completed trade"""
        # Update counters
        trades_total.labels(
            strategy_name=self.strategy_name,
            symbol=self.symbol, 
            trade_type=trade_type
        ).inc()
        
        # Record P&L distribution
        trade_pnl.labels(
            strategy_name=self.strategy_name,
            symbol=self.symbol
        ).observe(pnl)
        
        # Update position tracking
        position_size.labels(
            strategy_name=self.strategy_name,
            symbol=self.symbol,
            side=trade_type
        ).set(shares)
        
        self.trade_count += 1
        self.total_pnl += pnl
        
        print(f"📊 Trade recorded: {trade_type} {shares} shares, P&L: ${pnl:.2f}")
    
    def update_portfolio_value(self, current_value: float):
        """Update current portfolio value and drawdown metrics"""
        portfolio_value.labels(
            strategy_name=self.strategy_name,
            symbol=self.symbol
        ).set(current_value)
        
        # Calculate drawdown
        if current_value > self.peak_value:
            self.peak_value = current_value
            self.current_drawdown = 0.0
        else:
            self.current_drawdown = (self.peak_value - current_value) / self.peak_value
            if self.current_drawdown > self.max_drawdown:
                self.max_drawdown = self.current_drawdown
                
        # Update max drawdown metric
        strategy_max_drawdown.labels(
            strategy_name=self.strategy_name,
            symbol=self.symbol,
            timeframe=self.timeframe
        ).set(self.max_drawdown * 100)
    
    def finalize_backtest(self, initial_capital: float):
        """Record final backtest results"""
        execution_time = time.time() - self.backtest_start_time
        
        # Record execution time
        backtest_execution_time.labels(
            strategy_name=self.strategy_name,
            symbol=self.symbol,
            timeframe=self.timeframe
        ).observe(execution_time)
        
        # Calculate and record total return
        total_return_pct = (self.total_pnl / initial_capital) * 100
        strategy_total_return.labels(
            strategy_name=self.strategy_name,
            symbol=self.symbol,
            timeframe=self.timeframe
        ).set(total_return_pct)
        
        # Calculate and record Sharpe ratio (simplified)
        # In real implementation, this would use actual returns series
        sharpe_approx = total_return_pct / (self.max_drawdown * 100 + 1)  # Simplified calculation
        strategy_sharpe_ratio.labels(
            strategy_name=self.strategy_name,
            symbol=self.symbol,
            timeframe=self.timeframe
        ).set(sharpe_approx)
        
        # Record processing speed
        orders_per_second = self.trade_count / execution_time if execution_time > 0 else 0
        orders_processed_per_second.labels(
            strategy_name=self.strategy_name
        ).set(orders_per_second)
        
        print(f"✅ Backtest completed in {execution_time:.2f} seconds")
        print(f"📈 Total Return: {total_return_pct:.2f}%")
        print(f"📉 Max Drawdown: {self.max_drawdown*100:.2f}%") 
        print(f"⚡ Processing Speed: {orders_per_second:.2f} trades/second")

# Demo: Simulate a backtest with metrics collection
def simulate_backtest_with_metrics():
    """Simulate a backtest run with Prometheus metrics collection"""
    collector = BacktestMetricsCollector("MovingAverageCrossover", "BTCUSD", "1h")
    initial_capital = 100000.0
    current_portfolio = initial_capital
    
    print("🚀 Starting backtest simulation with metrics collection...")
    
    # Simulate 100 trades over time
    for i in range(100):
        # Simulate trade execution
        trade_type = random.choice(["BUY", "SELL"])
        shares = random.randint(1, 10)
        pnl = random.uniform(-200, 300)  # Random P&L between -$200 and +$300
        
        collector.record_trade(trade_type, pnl, shares)
        current_portfolio += pnl
        collector.update_portfolio_value(current_portfolio)
        
        # Small delay to simulate time passage
        time.sleep(0.01)
    
    collector.finalize_backtest(initial_capital)
    return collector

# Run the simulation
metrics_collector = simulate_backtest_with_metrics()

## 4. Configure Prometheus Scraping

Set up Prometheus configuration to scrape metrics from the backtesting engine and store time-series data.

In [ ]:
# Start Prometheus HTTP server to expose metrics
PROMETHEUS_PORT = 8000

try:
    # Start metrics server (this would normally run in your backtesting application)
    start_http_server(PROMETHEUS_PORT)
    print(f"✅ Prometheus metrics server started on port {PROMETHEUS_PORT}")
    print(f"🔍 Metrics endpoint: http://localhost:{PROMETHEUS_PORT}/metrics")
    
    # Show sample metrics output
    metrics_output = generate_latest().decode('utf-8')
    print("📊 Sample metrics being exposed:")
    print("=" * 60)
    
    # Filter and display relevant backtest metrics
    for line in metrics_output.split('\n'):
        if 'backtest_' in line and not line.startswith('#'):
            print(line)
            
except OSError as e:
    if "Address already in use" in str(e):
        print(f"⚠️  Port {PROMETHEUS_PORT} already in use (metrics server likely already running)")
        print(f"🔍 Metrics endpoint: http://localhost:{PROMETHEUS_PORT}/metrics")
    else:
        print(f"❌ Error starting metrics server: {e}")

In [ ]:
# Prometheus configuration for scraping BacktestingEngine metrics
prometheus_config = """
# Prometheus configuration for BacktestingEngine
global:
  scrape_interval: 15s
  evaluation_interval: 15s

rule_files:
  - "backtesting_alerts.yml"

scrape_configs:
  # BacktestingEngine metrics
  - job_name: 'backtesting-engine'
    static_configs:
      - targets: ['localhost:8000']  # BacktestingEngine metrics endpoint
    scrape_interval: 5s
    scrape_timeout: 4s
    metrics_path: '/metrics'
    
  # Kubernetes pod scraping (when deployed)
  - job_name: 'kubernetes-pods'
    kubernetes_sd_configs:
      - role: pod
    relabel_configs:
      - source_labels: [__meta_kubernetes_pod_annotation_prometheus_io_scrape]
        action: keep
        regex: true
      - source_labels: [__meta_kubernetes_pod_annotation_prometheus_io_path]
        action: replace
        target_label: __metrics_path__
        regex: (.+)
      - source_labels: [__address__, __meta_kubernetes_pod_annotation_prometheus_io_port]
        action: replace
        regex: ([^:]+)(?::\\d+)?;(\\d+)
        replacement: $1:$2
        target_label: __address__

alerting:
  alertmanagers:
    - static_configs:
        - targets:
          - alertmanager:9093
"""

print("📋 Prometheus Configuration for BacktestingEngine:")
print("=" * 60)
print(prometheus_config)

# Save configuration to file
with open('C:\\Users\\ikenn\\Projects\\TradingPlatform\\BacktestingEngine\\prometheus.yml', 'w') as f:
    f.write(prometheus_config)

print("\n✅ Prometheus configuration saved to BacktestingEngine/prometheus.yml")

## 5. Create Grafana Dashboard Programmatically

Use Grafana API and Python to programmatically create dashboards for visualizing backtesting results, including charts for equity curves, drawdowns, and trade statistics.

In [ ]:
# Grafana Dashboard Configuration for BacktestingEngine
grafana_dashboard = {
    "dashboard": {
        "id": None,
        "title": "BacktestingEngine Performance Dashboard",
        "tags": ["backtesting", "trading", "performance"],
        "timezone": "browser",
        "panels": [
            {
                "id": 1,
                "title": "Strategy Performance Overview",
                "type": "stat",
                "targets": [
                    {
                        "expr": "backtest_strategy_total_return_percent",
                        "legendFormat": "Total Return %",
                        "refId": "A"
                    },
                    {
                        "expr": "backtest_strategy_sharpe_ratio", 
                        "legendFormat": "Sharpe Ratio",
                        "refId": "B"
                    },
                    {
                        "expr": "backtest_strategy_max_drawdown_percent",
                        "legendFormat": "Max Drawdown %",
                        "refId": "C"
                    }
                ],
                "fieldConfig": {
                    "defaults": {
                        "color": {"mode": "palette-classic"},
                        "unit": "percent",
                        "decimals": 2
                    }
                },
                "gridPos": {"h": 8, "w": 24, "x": 0, "y": 0}
            },
            {
                "id": 2,
                "title": "Trade Execution Rate",
                "type": "graph",
                "targets": [
                    {
                        "expr": "rate(backtest_trades_total[1m])",
                        "legendFormat": "Trades/minute - {{trade_type}}",
                        "refId": "A"
                    }
                ],
                "yAxes": [
                    {
                        "label": "Trades per minute",
                        "min": 0
                    }
                ],
                "gridPos": {"h": 8, "w": 12, "x": 0, "y": 8}
            },
            {
                "id": 3,
                "title": "Portfolio Value Timeline", 
                "type": "graph",
                "targets": [
                    {
                        "expr": "backtest_portfolio_value_dollars",
                        "legendFormat": "Portfolio Value - {{symbol}}",
                        "refId": "A"
                    }
                ],
                "yAxes": [
                    {
                        "label": "Portfolio Value ($)",
                        "min": 0
                    }
                ],
                "gridPos": {"h": 8, "w": 12, "x": 12, "y": 8}
            },
            {
                "id": 4,
                "title": "Trade P&L Distribution",
                "type": "heatmap",
                "targets": [
                    {
                        "expr": "increase(backtest_trade_pnl_dollars_bucket[1h])",
                        "legendFormat": "{{le}}",
                        "refId": "A"
                    }
                ],
                "gridPos": {"h": 8, "w": 24, "x": 0, "y": 16}
            },
            {
                "id": 5,
                "title": "Backtest Execution Performance",
                "type": "graph",
                "targets": [
                    {
                        "expr": "backtest_execution_duration_seconds",
                        "legendFormat": "Execution Time - {{strategy_name}}",
                        "refId": "A"
                    },
                    {
                        "expr": "backtest_orders_processed_per_second",
                        "legendFormat": "Orders/sec - {{strategy_name}}",
                        "refId": "B"
                    }
                ],
                "gridPos": {"h": 8, "w": 24, "x": 0, "y": 24}
            }
        ],
        "time": {
            "from": "now-1h",
            "to": "now"
        },
        "refresh": "10s"
    }
}

def create_grafana_dashboard(grafana_url="http://localhost:3000", api_key=None):
    """Create Grafana dashboard programmatically"""
    
    headers = {
        "Content-Type": "application/json"
    }
    
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    else:
        # For demo purposes, we'll just print the configuration
        print("🎨 Grafana Dashboard Configuration for BacktestingEngine:")
        print("=" * 60)
        print(json.dumps(grafana_dashboard, indent=2))
        print("\n💡 To create this dashboard:")
        print("1. Go to your Grafana instance")
        print("2. Navigate to '+' -> Import") 
        print("3. Paste the JSON configuration above")
        print("4. Configure data source as your Prometheus instance")
        return
    
    try:
        response = requests.post(
            f"{grafana_url}/api/dashboards/db",
            headers=headers,
            json=grafana_dashboard,
            timeout=10
        )
        
        if response.status_code == 200:
            print("✅ Grafana dashboard created successfully!")
            print(f"🔗 Dashboard URL: {grafana_url}/d/{response.json()['uid']}")
        else:
            print(f"❌ Failed to create dashboard: {response.status_code} - {response.text}")
            
    except requests.exceptions.RequestException as e:
        print(f"❌ Error connecting to Grafana: {e}")

# Create the dashboard (without API key for demo)
create_grafana_dashboard()

## 6. Real-time Backtesting Monitoring

Implement real-time monitoring of backtesting runs with alerts and notifications for significant events or performance thresholds.

In [ ]:
# Alerting rules for BacktestingEngine
alerting_rules = """
groups:
  - name: backtesting_alerts
    rules:
    
    # Strategy Performance Alerts
    - alert: PoorStrategyPerformance
      expr: backtest_strategy_total_return_percent < -10
      for: 1m
      labels:
        severity: warning
        component: backtesting
      annotations:
        summary: "Strategy {{ $labels.strategy_name }} showing poor performance"
        description: "Strategy {{ $labels.strategy_name }} on {{ $labels.symbol }} has total return of {{ $value }}%"
    
    - alert: ExcessiveDrawdown
      expr: backtest_strategy_max_drawdown_percent > 20
      for: 30s
      labels:
        severity: critical
        component: backtesting
      annotations:
        summary: "Excessive drawdown detected"
        description: "Strategy {{ $labels.strategy_name }} has maximum drawdown of {{ $value }}%"
    
    # Execution Performance Alerts
    - alert: SlowBacktestExecution
      expr: backtest_execution_duration_seconds > 300
      for: 10s
      labels:
        severity: warning
        component: performance
      annotations:
        summary: "Slow backtest execution detected"
        description: "Backtest for {{ $labels.strategy_name }} took {{ $value }} seconds to complete"
    
    - alert: LowThroughput
      expr: backtest_orders_processed_per_second < 10
      for: 1m
      labels:
        severity: warning
        component: performance
      annotations:
        summary: "Low order processing throughput"
        description: "Only {{ $value }} orders/second being processed for {{ $labels.strategy_name }}"
    
    # Trade Execution Alerts
    - alert: NoTradesExecuted
      expr: increase(backtest_trades_total[5m]) == 0
      for: 2m
      labels:
        severity: warning
        component: trading
      annotations:
        summary: "No trades executed recently"
        description: "Strategy {{ $labels.strategy_name }} has not executed any trades in the last 5 minutes"
    
    - alert: LargeLoss
      expr: backtest_trade_pnl_dollars < -1000
      for: 0s
      labels:
        severity: critical
        component: trading
      annotations:
        summary: "Large trade loss detected"
        description: "Strategy {{ $labels.strategy_name }} had a trade loss of ${{ $value }}"
"""

print("🚨 Prometheus Alerting Rules for BacktestingEngine:")
print("=" * 60)
print(alerting_rules)

# Save alerting rules
with open('C:\\Users\\ikenn\\Projects\\TradingPlatform\\BacktestingEngine\\backtesting_alerts.yml', 'w') as f:
    f.write(alerting_rules)

print("\n✅ Alerting rules saved to BacktestingEngine/backtesting_alerts.yml")

In [ ]:
# Real-time monitoring dashboard with Python
def create_realtime_monitoring_dashboard():
    """Create a simple real-time monitoring dashboard using Plotly"""
    
    # Generate sample time series data for demonstration
    timestamps = pd.date_range(start='2024-01-01', periods=100, freq='H')
    
    # Simulate portfolio value over time
    portfolio_values = []
    initial_value = 100000
    current_value = initial_value
    
    for i in range(len(timestamps)):
        # Simulate market movements
        change = random.uniform(-0.02, 0.03)  # -2% to +3% change
        current_value *= (1 + change)
        portfolio_values.append(current_value)
    
    # Calculate drawdown
    peak_values = pd.Series(portfolio_values).cummax()
    drawdowns = (peak_values - portfolio_values) / peak_values * 100
    
    # Create subplots
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=(
            'Portfolio Value Over Time', 'Drawdown Analysis',
            'Trade P&L Distribution', 'Execution Performance',
            'Trade Volume by Type', 'Risk Metrics'
        ),
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # Portfolio value chart
    fig.add_trace(
        go.Scatter(x=timestamps, y=portfolio_values, 
                  name='Portfolio Value', line=dict(color='green')),
        row=1, col=1
    )
    
    # Drawdown chart  
    fig.add_trace(
        go.Scatter(x=timestamps, y=drawdowns,
                  name='Drawdown %', fill='tozeroy', 
                  line=dict(color='red')),
        row=1, col=2
    )
    
    # Trade P&L histogram
    trade_pnls = [random.uniform(-500, 800) for _ in range(50)]
    fig.add_trace(
        go.Histogram(x=trade_pnls, name='Trade P&L', 
                    marker_color='blue', opacity=0.7),
        row=2, col=1
    )
    
    # Execution time metrics
    execution_times = [random.uniform(0.1, 2.0) for _ in range(20)]
    execution_timestamps = timestamps[-20:]
    fig.add_trace(
        go.Bar(x=execution_timestamps, y=execution_times,
              name='Execution Time (s)', marker_color='orange'),
        row=2, col=2
    )
    
    # Trade volume by type
    trade_types = ['BUY', 'SELL']
    trade_counts = [45, 55]
    fig.add_trace(
        go.Pie(labels=trade_types, values=trade_counts, name='Trade Types'),
        row=3, col=1
    )
    
    # Risk metrics gauge
    sharpe_ratio = 1.8
    fig.add_trace(
        go.Indicator(
            mode="gauge+number",
            value=sharpe_ratio,
            title={'text': "Sharpe Ratio"},
            gauge={
                'axis': {'range': [None, 3]},
                'bar': {'color': "darkblue"},
                'steps': [
                    {'range': [0, 1], 'color': "lightgray"},
                    {'range': [1, 2], 'color': "yellow"},
                    {'range': [2, 3], 'color': "green"}
                ],
                'threshold': {
                    'line': {'color': "red", 'width': 4},
                    'thickness': 0.75,
                    'value': 2.5
                }
            }
        ),
        row=3, col=2
    )
    
    # Update layout
    fig.update_layout(
        title_text="BacktestingEngine - Real-time Monitoring Dashboard",
        height=900,
        showlegend=True
    )
    
    return fig

# Create and display the dashboard
monitoring_dashboard = create_realtime_monitoring_dashboard()
monitoring_dashboard.show()

print("📊 Real-time monitoring dashboard created!")
print("💡 This demonstrates how to visualize backtesting data in real-time")
print("🔄 In production, this would connect to live Prometheus metrics")

## Summary: Prometheus/Grafana vs. Specialized Analytics

### ✅ **Use Prometheus/Grafana for:**
- **Operational Monitoring**: Backtest execution performance, system resources, error rates
- **Real-time Alerts**: Performance degradation, system failures, threshold breaches  
- **Infrastructure Metrics**: Kubernetes pod health, database connections, memory usage
- **Continuous Integration**: Monitoring automated backtest pipelines

### ❌ **Don't Use Prometheus/Grafana for:**
- **Financial Analysis**: Detailed equity curves, risk attribution, statistical analysis
- **Trade-by-Trade Analysis**: Individual trade performance, order flow analysis
- **Research & Development**: Strategy optimization, parameter sensitivity analysis
- **Regulatory Reporting**: Compliance reports, audit trails, detailed performance attribution

### 💡 **Recommended Hybrid Approach:**

1. **Prometheus/Grafana**: Operations & monitoring (as shown above)
2. **Jupyter + Plotly/Matplotlib**: Financial analysis & research  
3. **Database**: Store detailed backtest results in PostgreSQL
4. **Custom Reports**: Generate PDF/HTML reports for stakeholders

### 🎯 **Best Practice Architecture:**

```
BacktestingEngine
├── Prometheus Metrics (Operations)
├── Database Storage (Detailed Results) 
├── Jupyter Notebooks (Analysis)
├── Custom Reports (Stakeholders)
└── Grafana Dashboards (Real-time Monitoring)
```

This gives you the best of both worlds - operational excellence through Prometheus/Grafana and deep financial analysis through specialized tools.